In [ ]:
SMOKE = True   # True = smoke run: 8 pairs per scoring set, 3 warm-up + 8 latency samples, 2 throughput
               # batches, every output under /kaggle/working/smoke/, every number DISCARDED.
               # Set to False for the full run (docs/phase3-llm-int8-variants-runbook.md).
# PricePilot Phase 3 item 8 session 2 task 3 -- LLM int8 variants V2/V3 (CE V2 is deferred, NOT
# built here -- protocol 5.11). Protocol: docs/phase3-serving-benchmark-protocol.md section 5.11
# (pre-registered BEFORE this notebook existed). This notebook computes NO F1 and reads NO TEST
# label: it only writes scores, latencies and facts. Threshold selection, eligibility and the
# variant decision happen locally, in session 2, from the files this notebook writes.
N_SET = 8 if SMOKE else None          # pairs per scoring set (None = all)
N_WARMUP = 3 if SMOKE else 10         # untimed warm-up pairs before the latency pass
N_LAT = 8 if SMOKE else 287           # latency samples (protocol 5.5: the 287 TEST pairs)
N_THR_BATCHES = 2 if SMOKE else 10    # throughput: batches of 16 (protocol 5.5: 160 pairs)
THR_BATCH = 16


In [ ]:
import ast, contextlib, gc, glob, hashlib, json, os, platform, re, resource, shutil, subprocess, sys, time, traceback
from pathlib import Path

WORK = Path("/kaggle/working")
OUT = WORK / "smoke" if SMOKE else WORK
TMP = (Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path("/tmp")) / ("pp-smoke" if SMOKE else "pp-full")
OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)   # ONNX graphs live here, NOT in the saved output

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript", "peft"],
                   capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])
assert r.returncode == 0, "pip install failed"

import numpy as np
import torch
import transformers
import onnx, onnxruntime, peft, tokenizers

from typing import Any  # only for the ENV-COMPAT block below

# Ported verbatim from notebooks/phase3-llm-finetune.py's own ENV-COMPAT-BEGIN/END block;
# byte-identity checked by tests/test_torchao_env_compat_notebook_parity.py. Kaggle's image
# ships torchao 0.10.0 but the installed peft only supports torchao >= 0.16.0, so
# llm_merge (below) is the first thing that calls into peft's from_pretrained loader, and
# without this patch that call raises ImportError from
# peft/tuners/lora/torchao.py::dispatch_torchao. Runs here, right after peft is imported.
# ENV-COMPAT-BEGIN
# ---------------------------------------------------------------------------
# ENVIRONMENT COMPATIBILITY -- Kaggle's image ships torchao 0.10.0, but the installed peft only
# supports torchao >= 0.16.0 and RAISES ImportError (instead of returning False) from
# is_torchao_available() when it probes an older one, inside get_peft_model ->
# peft/tuners/lora/torchao.py::dispatch_torchao. This project does not use torchao at all; peft
# only probes it while choosing which layer class to wrap. So: try to uninstall it, then force the
# probe to report False. BOTH bindings are patched: peft.tuners.lora.torchao imports the name
# `is_torchao_available` into its own namespace at import time, so patching peft.import_utils
# alone would leave the old reference in place and the raise would still happen.
# On a future image without the problem this block is a harmless no-op (the patched probe just
# says "no torchao", which is true for our purposes).
# ---------------------------------------------------------------------------
# Best effort only; the patch below is what guarantees safety.
_uninstall_rc: int | str = "not attempted"
with contextlib.suppress(Exception):
    _uninstall_rc = subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        capture_output=True,
        timeout=120,
        check=False,
    ).returncode
print(f"ENV COMPAT: pip uninstall torchao return code: {_uninstall_rc}")

try:
    import peft.import_utils
    import peft.tuners.lora.torchao as _lora_torchao

    def _no_torchao(*_a: Any, **_k: Any) -> bool:
        return False

    peft.import_utils.is_torchao_available = _no_torchao  # type: ignore[assignment]
    _patched = ["peft.import_utils"]
    if hasattr(_lora_torchao, "is_torchao_available"):
        _lora_torchao.is_torchao_available = _no_torchao  # type: ignore[assignment]
        _patched.append("peft.tuners.lora.torchao")
    print(
        f"ENV COMPAT: peft's torchao probe NEUTRALISED (is_torchao_available -> False in "
        f"{' and '.join(_patched)}). Reason: Kaggle ships torchao 0.10.0, peft requires >= 0.16.0 "
        "and raises instead of returning False. Expected; not used here."
    )
    if len(_patched) < 2:
        print("ENV COMPAT: WARNING -- lora.torchao has no is_torchao_available binding to patch.")
except Exception as _compat_err:
    print(f"ENV COMPAT: torchao probe patch not applied ({_compat_err!r}); continuing.")
# ENV-COMPAT-END

def jdump(obj, name, folder=None):
    path = (folder or OUT) / name
    path.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    return path

def jupdate(name, key, value):
    path = OUT / name
    d = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    d[key] = value
    path.write_text(json.dumps(d, indent=1), encoding="utf-8")

STATUS = []
STATUS_NOTE = {}   # health notes written by a stage (informational; the gates run locally)
def run_stage(name, fn):
    """Every stage: try/except, outputs written as soon as it has them, one status row appended."""
    t0 = time.time(); ok, err = True, None
    try:
        fn()
    except (Exception, SystemExit):
        ok, err = False, traceback.format_exc()
        print(f"[{name}] FAILED\n{err}")
    STATUS.append({"stage": name, "ok": ok, "error": err, "seconds": round(time.time() - t0, 1)})
    if name in STATUS_NOTE:
        STATUS[-1]["note"] = STATUS_NOTE[name]
    jdump(STATUS, "stage-status.json")
    print(f"[{name}] ok={ok} {STATUS[-1]['seconds']}s")
    gc.collect()

def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise SystemExit(f"input not found: {what} (glob {pattern}). Add it under Add Input.")
    if len(hits) > 1:
        print(f"WARNING: {len(hits)} matches for {what}; using {hits[0]}: {hits}")
    return Path(hits[0])

def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

def dir_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file())

# ---------- env-facts.json ----------
lscpu = subprocess.run(["bash", "-c", "lscpu"], capture_output=True, text=True).stdout
meminfo = Path("/proc/meminfo").read_text().splitlines()[0]
PIN_CPUS = sorted(os.sched_getaffinity(0))[:2]
jdump({
    "smoke": SMOKE,
    "lscpu_model_name": next((l.split(":", 1)[1].strip() for l in lscpu.splitlines() if l.startswith("Model name")), None),
    "lscpu": lscpu, "cpu_count": os.cpu_count(), "affinity_at_start": sorted(os.sched_getaffinity(0)),
    "cpu_vnni": (lambda fl: None if fl is None else {f: f in fl for f in ("avx2", "avx512f", "avx512_vnni", "avx_vnni")})(
        next((l.split(":", 1)[1].split() for l in lscpu.splitlines() if l.startswith("Flags")), None)),   # None = no Flags line
    "pinned_cpus_for_latency": PIN_CPUS, "mem_total": meminfo,
    "thread_siblings": {c: Path(f"/sys/devices/system/cpu/cpu{c}/topology/thread_siblings_list").read_text().strip()
                        for c in sorted(os.sched_getaffinity(0)) if Path(f"/sys/devices/system/cpu/cpu{c}/topology/thread_siblings_list").exists()},
    "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__, "onnx": onnx.__version__,
    "onnxruntime": onnxruntime.__version__, "peft": peft.__version__, "tokenizers": tokenizers.__version__,
    "numpy": np.__version__,
    "optimum": "not used: the LoRA graph is custom (two readout logits only), so the export is torch.onnx.export",
}, "env-facts.json")

# ---------- inputs, located by glob, no slug hardcoded ----------
TEST = json.loads(find_one("/kaggle/input/**/phase3-inputs-test.json", "inputs dataset (test)").read_text(encoding="utf-8"))
TV = json.loads(find_one("/kaggle/input/**/phase3-inputs-train-val.json", "inputs dataset (train-val)").read_text(encoding="utf-8"))
assert TEST["frozen_labels_sha256"] == TV["frozen_labels_sha256"]
test_rows = sorted(TEST["pairs"], key=lambda r: r["pair_id"])
tv_rows = sorted(TV["pairs"], key=lambda r: r["pair_id"])
assert len(test_rows) == 287 and len(tv_rows) == 672
assert not any(k in r for r in test_rows for k in ("label", "tier", "split")), "TEST leak!"
# TRAIN_VAL labels are used ONLY to pick which pair_ids are the 133 scored validation pairs.
val_rows = [r for r in tv_rows if r["train_or_val"] == "val" and r["label"] in ("M", "N")]
assert len(val_rows) == 133, len(val_rows)

def head(rows):
    return rows[:N_SET] if N_SET else rows

# No CE weights input in this notebook (only 2 inputs -- runbook: phase3-inputs dataset +
# notebookf26a8565eb); the CE V2 robustness check is deferred (protocol 5.11).
LLM_SETS = {"val": head(val_rows), "test": head(test_rows)}
print({k: len(v) for k, v in LLM_SETS.items()})

BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
S = {}   # cross-stage state

In [ ]:
from typing import Any
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

# TEMPLATE-BEGIN
# Kept byte-identical in notebooks/phase3-llm-finetune.py; checked by
# tests/test_llm_prompt_notebook_parity.py, which extracts the text strictly between the
# "# TEMPLATE-BEGIN" and "# TEMPLATE-END" marker lines in both files and asserts equality.
# Nothing outside these two marker lines is compared, so comments/docstrings elsewhere in either
# file may differ freely -- only what sits between them is the contract.

# Bumped whenever the template text changes, so a stale comparison (baseline scored under one
# template, notebook trained under a byte-different one) can never happen silently. Recorded in
# DECISIONS.md ADR-0028 addendum #21 and printed by the notebook.
LLM_PROMPT_VERSION = "llm-prompt-v1"

# Deliberately silent on tiers, split membership, label counts, or how the dataset was sampled --
# none of that is information the task itself needs, and stating it would let the model condition
# on sampling design rather than on the two listings' actual content. States the same operational
# question and rule ladder docs/learned/phase3-annotation-conventions.md defines for the human
# annotator (rules 1-6), so the model is asked the same question the labels themselves answer --
# not a looser or stricter one.
_INSTRUCTION = (
    "You are comparing two competitor-shop listings for pet food, treats or litter products.\n"
    "Decide whether Listing A and Listing B are the SAME purchasable unit: a price-comparison "
    "engine comparing their prices would be comparing the same physical product.\n"
    "\n"
    "Each of these, on its own, makes them NOT the same unit: a different net weight or volume; "
    "a different pack count (a stated count of 1 and no count stated are the SAME); a different "
    "bonus weight; a different species (dog vs cat); a different life-stage, sterilised, light, "
    "sensitive, indoor, dental, urinary or joint-care formula; a different food form (dry vs "
    "wet/tin/pouch -- wet, tin and pouch among themselves are NOT a difference); a different "
    "breed size (e.g. Mini vs Maxi); a different flavour (the same flavour word in another "
    "language, e.g. Salmon/Somon, is NOT a difference).\n"
    "\n"
    "None of these, on their own, make them different: generic descriptive words (grain-free, "
    "premium, natural, holistic); an animal-weight dosage band (e.g. 12-25 kg) stated on only "
    "one side; a mismatched brand name alone when everything else about the listing matches.\n"
    "\n"
    "Answer with exactly one word: Yes or No."
)


def build_llm_prompt(text_a: str, text_b: str) -> str:
    """Build the completion prompt; see module docstring for the design rationale."""
    return f"{_INSTRUCTION}\n\nListing A: {text_a}\nListing B: {text_b}\nAnswer:"


# TEMPLATE-END

# Readout token ids, resolved EXACTLY as notebooks/phase3-llm-finetune.py resolves them.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def _single_token_id(word: str) -> int:
    ids = tokenizer.encode(word, add_special_tokens=False)
    if len(ids) != 1:
        raise SystemExit(
            f"REFUSING TO RUN: {word!r} encodes to {len(ids)} token(s) ({ids}), not exactly "
            "one, for this tokenizer -- the two-token readout requires both ' Yes' and ' No' to "
            "be single tokens. Falling back silently would score gibberish; fix the readout "
            "design (a different pair of words, or a different model) instead of proceeding."
        )
    return ids[0]


YES_TOKEN_ID = _single_token_id(" Yes")
NO_TOKEN_ID = _single_token_id(" No")

def llm_rows(rows):
    return [{"pair_id": r["pair_id"], "prompt": build_llm_prompt(r["text_a"], r["text_b"])} for r in rows]

def ce_rows(rows):
    return [{"pair_id": r["pair_id"], "text_a": r["text_a"], "text_b": r["text_b"]} for r in rows]

In [ ]:
# The worker is a separate process: OMP_NUM_THREADS / CPU affinity must be set BEFORE onnxruntime is
# imported, and peak RSS is only meaningful per model if the process holds nothing but that model.
# It imports onnxruntime + numpy + tokenizers only (no torch): this is the serving path. ru_maxrss is
# process-lifetime, so EVERY latency/throughput/RSS measurement runs in its own fresh child that loads
# exactly one ORT session; the parent's PyTorch reference models never share a process with it.
WORKER_SRC = r"""
import json, os, sys, time
job = json.load(open(sys.argv[1]))
PIN = job["pin"]
# Overridable only so the local test suite (no /proc on Windows) can point this at a fixture
# file; on Kaggle job never sets it and this is always the real "/proc/self/status".
PROC_STATUS_PATH = job.get("proc_status_path", "/proc/self/status")
if PIN:
    for k in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
        os.environ[k] = "2"
    cpus = sorted(os.sched_getaffinity(0))[:2]
    os.sched_setaffinity(0, cpus)
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer

# ru_maxrss is preserved across fork+execve on Linux, so a child started by subprocess.run
# from the parent notebook process inherits the PARENT process peak instead of its own.
# VmHWM (peak) / VmRSS (current), read fresh from /proc/self/status, belong to the mm this
# process got at exec, so they are this child, and only this child.
def proc_rss_mb():
    hwm = cur = None
    with open(PROC_STATUS_PATH) as f:
        for line in f:
            if line.startswith("VmHWM:"):
                hwm = float(line.split()[1]) / 1024.0
            elif line.startswith("VmRSS:"):
                cur = float(line.split()[1]) / 1024.0
    return hwm, cur

so = ort.SessionOptions()
if PIN:
    so.intra_op_num_threads = 2
    so.inter_op_num_threads = 1
    so.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
sess = ort.InferenceSession(job["model"], so, providers=["CPUExecutionProvider"])
RSS_AFTER_LOAD, CUR_RSS_AFTER_LOAD = proc_rss_mb()   # this child only
assert [i.name for i in sess.get_inputs()] == ["input_ids", "attention_mask"], sess.get_inputs()
tok = Tokenizer.from_file(job["tokenizer"])
tok.no_padding()
KIND, PAD, MAXLEN = job["kind"], job["pad_id"], 256
if KIND == "ce":
    tok.enable_truncation(max_length=MAXLEN)

def tokenize(row):
    if KIND == "ce":
        return tok.encode(row["text_a"], row["text_b"]).ids
    ids = tok.encode(row["prompt"], add_special_tokens=False).ids
    assert len(ids) <= 1024, "prompt over the training MAX_LENGTH: refusing to truncate"
    return ids

def batch_of(id_lists):
    n = max(len(x) for x in id_lists)
    ids = np.full((len(id_lists), n), PAD, dtype=np.int64)
    mask = np.zeros((len(id_lists), n), dtype=np.int64)
    for i, x in enumerate(id_lists):
        ids[i, : len(x)] = x
        mask[i, : len(x)] = 1
    return {"input_ids": ids, "attention_mask": mask}

def readout(out):
    out = out.astype(np.float64)
    if KIND == "ce":
        return (1.0 / (1.0 + np.exp(-out[:, 0]))).tolist()
    return (1.0 / (1.0 + np.exp(out[:, 1] - out[:, 0]))).tolist()   # softmax over (Yes, No), P(Yes)

def score_one(row):
    return readout(sess.run(None, batch_of([tokenize(row)]))[0])[0]

result = {}
if job.get("score"):
    for name, spec in job["score"].items():
        rows = json.load(open(spec["pairs"]))
        scores = {r["pair_id"]: score_one(r) for r in rows}
        json.dump(scores, open(spec["out"], "w"), indent=1)
        result[name] = len(scores)

def pct(a):
    a = np.asarray(a, dtype=np.float64)
    return {"p50": float(np.percentile(a, 50)), "p95": float(np.percentile(a, 95)),
            "p99": float(np.percentile(a, 99)), "mean": float(a.mean()), "n": int(a.size)}

if job.get("bench"):
    b = job["bench"]
    rows = json.load(open(b["pairs"]))
    for r in rows[: b["n_warmup"]]:
        score_one(r)
    e2e, run_only, lens = [], [], []
    for r in rows[: b["n_lat"]]:
        t0 = time.perf_counter_ns()
        ids = tokenize(r)
        feed = batch_of([ids])
        t1 = time.perf_counter_ns()
        out = sess.run(None, feed)[0]
        t2 = time.perf_counter_ns()
        readout(out)
        t3 = time.perf_counter_ns()
        e2e.append((t3 - t0) / 1e6)
        run_only.append((t2 - t1) / 1e6)
        lens.append(len(ids))
    lat = {
        "model": job["name"], "variant": job["variant"], "pinned": PIN,
        "threads": 2 if PIN else None, "cpus": sorted(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
        "measured_in_fresh_process_pid": os.getpid(), "peak_rss_mb_after_session_load": RSS_AFTER_LOAD,
        "current_rss_mb_after_session_load": CUR_RSS_AFTER_LOAD,
        "ort_version": ort.__version__, "n_warmup": b["n_warmup"],
        "summary_e2e_ms": pct(e2e), "summary_run_only_ms": pct(run_only),
        "token_length": {"min": int(min(lens)), "p50": float(np.percentile(lens, 50)),
                         "p95": float(np.percentile(lens, 95)), "max": int(max(lens))},
        "samples_e2e_ms": e2e, "samples_run_only_ms": run_only, "token_lengths": lens,
    }
    rss_after_latency, cur_rss_after_latency = proc_rss_mb()
    # throughput: batches of 16, tokenize + pad + run + readout, wall clock
    bs, batch_secs = b["batch"], []
    for i in range(b["n_thr_batches"]):
        chunk = rows[i * bs : (i + 1) * bs]
        t0 = time.perf_counter()
        feed = batch_of([tokenize(r) for r in chunk])
        readout(sess.run(None, feed)[0])
        batch_secs.append(time.perf_counter() - t0)
    n_pairs = b["n_thr_batches"] * bs
    thr = {"pairs": n_pairs, "batch": bs, "batch_seconds": batch_secs,
           "pairs_per_s": n_pairs / sum(batch_secs)}
    lat["throughput"] = thr
    lat["peak_rss_mb_after_batch1_latency"] = rss_after_latency   # batch-1 serving
    lat["current_rss_mb_after_batch1_latency"] = cur_rss_after_latency
    lat["peak_rss_mb"], lat["current_rss_mb"] = proc_rss_mb()   # after the batch-16 pass
    json.dump(lat, open(b["out"], "w"), indent=1)
    result["bench"] = {"p50_e2e_ms": lat["summary_e2e_ms"]["p50"], "pairs_per_s": thr["pairs_per_s"],
                       "peak_rss_mb": lat["peak_rss_mb"]}
print(json.dumps(result))
"""
(TMP / "worker.py").write_text(WORKER_SRC, encoding="utf-8")

def run_worker(tag, **job):
    path = TMP / f"job-{tag}.json"
    path.write_text(json.dumps(job), encoding="utf-8")
    r = subprocess.run([sys.executable, str(TMP / "worker.py"), str(path)], capture_output=True, text=True)
    print(f"[worker {tag}] rc={r.returncode} {r.stdout.strip()[-300:]}")
    if r.returncode != 0:
        raise RuntimeError(f"worker {tag} failed:\n{r.stderr[-2000:]}")

def write_rows(kind, name, rows):
    path = TMP / f"pairs-{kind}-{name}.json"
    path.write_text(json.dumps(ce_rows(rows) if kind == "ce" else llm_rows(rows)), encoding="utf-8")
    return str(path)

def score_spec(kind, sets, tag):
    """One scoring spec per set. Files: preds-<kind>-<tag>-<set>.json"""
    return {name: {"pairs": write_rows(kind, name, rows), "out": str(OUT / f"preds-{kind}-{tag}-{name}.json")}
            for name, rows in sets.items()}

def bench_spec(kind, name, variant):
    need = max(N_LAT, N_WARMUP, N_THR_BATCHES * THR_BATCH)   # test_rows is already in pair_id order
    return {"pairs": write_rows(kind, "bench", test_rows[:need]),
            "n_warmup": N_WARMUP, "n_lat": N_LAT, "n_thr_batches": N_THR_BATCHES, "batch": THR_BATCH,
            "out": str(OUT / f"latency-{name}-{variant}.json")}

def quantize(src, dst, external, **kw):
    """kw forwards onnxruntime.quantization.quantize_dynamic options -- protocol 5.11's V2
    (per_channel=True, reduce_range=True) and V3 (V2 + op_types_to_quantize=["MatMul"])."""
    from onnxruntime.quantization import QuantType, quantize_dynamic
    Path(dst).parent.mkdir(parents=True, exist_ok=True)
    quantize_dynamic(str(src), str(dst), weight_type=QuantType.QInt8, use_external_data_format=external, **kw)

def export_onnx(wrapper, example, path, dynamic_axes):
    """Legacy TorchScript exporter first (robust for BERT/Qwen); the dynamo exporter as fallback."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    kw = dict(input_names=["input_ids", "attention_mask"], output_names=["out"], dynamic_axes=dynamic_axes, opset_version=17)
    wrapper.eval()
    with torch.no_grad():
        try:
            torch.onnx.export(wrapper, example, str(path), dynamo=False, **kw)
        except Exception as e:
            print("legacy exporter failed, trying dynamo:", repr(e)[:300])
            torch.onnx.export(wrapper, example, str(path), dynamo=True, **kw)

In [ ]:
def llm_merge():
    """Find the adapter, prove which folder is which, merge into the fp32 base."""
    cfgs = sorted(glob.glob("/kaggle/input/**/adapter_config.json", recursive=True))
    dirs = {Path(c).parent.name: Path(c).parent for c in cfgs}
    print("adapter folders:", {k: str(v) for k, v in dirs.items()})
    hashes = {k: sha256_file(v / "adapter_model.safetensors") for k, v in dirs.items()}
    jupdate("model-facts.json", "lora_adapter_dirs", {k: {"path": str(v), "sha256": hashes[k]} for k, v in dirs.items()})
    if not dirs:
        raise SystemExit("no adapter_config.json under /kaggle/input")
    if len(set(hashes.values())) > 1:
        raise SystemExit(f"adapter folders differ, identity unresolved: {hashes}")
    adapter_dir = dirs.get("adapter") or dirs.get("adapter-epoch-8") or next(iter(dirs.values()))
    try:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float32)
    except TypeError:
        base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    from peft import PeftModel
    merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload().eval()
    assert next(merged.parameters()).dtype == torch.float32
    dst = TMP / "llm-merged"
    merged.save_pretrained(dst)
    tokenizer.save_pretrained(dst)
    jupdate("model-facts.json", "llm_param_count_merged", sum(p.numel() for p in merged.parameters()))
    jupdate("model-facts.json", "llm_adapter_used", str(adapter_dir))
    jupdate("model-facts.json", "llm_yes_no_token_ids", [YES_TOKEN_ID, NO_TOKEN_ID])
    S["llm_merged"] = dst

def llm_export():
    model = AutoModelForCausalLM.from_pretrained(S["llm_merged"], dtype=torch.float32, attn_implementation="eager").eval()
    class LLMWrap(torch.nn.Module):
        """Transformer body + ONLY the two readout logits at the last real prompt token.
        No KV-cache, one forward pass, no full-vocab logits (protocol 5.2)."""
        def __init__(self, lm, yes_id, no_id):
            super().__init__()
            self.body = lm.model
            self.register_buffer("head", lm.lm_head.weight[[yes_id, no_id]].detach().clone())
        def forward(self, input_ids, attention_mask):
            h = self.body(input_ids=input_ids, attention_mask=attention_mask, use_cache=False).last_hidden_state
            last = (attention_mask.sum(dim=1) - 1).view(-1, 1, 1).expand(-1, 1, h.size(-1))
            return h.gather(1, last).squeeze(1) @ self.head.T
    rows = LLM_SETS["test"][:2]
    enc = [tokenizer(build_llm_prompt(r["text_a"], r["text_b"]), add_special_tokens=False)["input_ids"] for r in rows]
    n = max(len(x) for x in enc)
    ids = torch.tensor([x + [tokenizer.pad_token_id] * (n - len(x)) for x in enc])
    mask = torch.tensor([[1] * len(x) + [0] * (n - len(x)) for x in enc])
    tmp_path = TMP / "llm-fp32-raw" / "model.onnx"
    export_onnx(LLMWrap(model, YES_TOKEN_ID, NO_TOKEN_ID), (ids, mask), tmp_path,
                {"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "out": {0: "b"}})
    # consolidate into ONE external-data file so the graph is a single small .onnx + one .data
    path = TMP / "llm-fp32" / "model.onnx"
    path.parent.mkdir(parents=True, exist_ok=True)
    m = onnx.load(str(tmp_path))
    onnx.save_model(m, str(path), save_as_external_data=True, all_tensors_to_one_file=True,
                    location="model.onnx.data", size_threshold=1024)
    del m
    shutil.rmtree(tmp_path.parent, ignore_errors=True)
    S["llm_fp32"] = path
    jupdate("model-facts.json", "llm_onnx_fp32", {"bytes": dir_bytes(path.parent), "sha256_graph": sha256_file(path),
                                                 "sha256_data": sha256_file(path.parent / "model.onnx.data")})

# Committed docs/learned/results/serving/preds-llm-onnxfp32-val.json (133 pairs, sha256
# 91f96e59e8531af6074b8529f960c33fcfc93a5c7ec3017f7e0cc075316c3ad6), embedded so this notebook can gate
# itself against it without a second Kaggle input dataset. This is the SAME merge (same
# adapter, same base) exported fresh in THIS run; if it does not reproduce the committed
# numbers closely, nothing downstream (V2/V3, built from THIS export) can be trusted.
COMMITTED_LLM_ONNXFP32_VAL = json.loads(r"""{"009591168d82_10e3017aee93":0.00016141092016798035,"0229d6ebd261_09ff7c94ad04":0.9980500485759429,"044ee1ecb78d_18118aeaeaaa":0.006116931346660093,"044ee1ecb78d_3df7a1df49ee":0.9831091113726383,"0571a46b10c4_494319b8d145":0.589431598308459,"085fe137cdf2_3df7a1df49ee":0.0010100068553380577,"085fe137cdf2_c55961beed26":0.9958115753263261,"08c409fce53f_8086ad9f717e":0.9811919128411903,"09aeddc5955c_ad5ae50fd254":0.9983842456628474,"09aeddc5955c_d01f3e239423":0.9980294581013803,"09ff7c94ad04_2ed57bcc51d8":0.00041274065524477655,"09ff7c94ad04_a8391a7250f9":0.00028013458076459454,"0aebed22fbf0_2f0ffa2e09f0":0.04315832896990178,"0b9df5e81782_15d4ac2e73f7":0.0007623727374425289,"0b9df5e81782_1d71ae73298c":0.0003054892815434138,"0b9df5e81782_272473d519f1":0.9918675253161618,"0b9df5e81782_6475e64e7766":0.0003185809550303198,"0ce14e895477_615bddaa6122":0.0004811201933273036,"0d1b18f0a21e_1d71ae73298c":0.9889942640192027,"0fb28ed4b16d_1155978dcb8f":0.9948064108054888,"0fb28ed4b16d_e6197dc16c1e":0.9918254311071666,"102e175bc774_905ae23f05fb":0.025972890036898084,"103e420055f5_9c67f10500d6":0.0008318356654644913,"14fc97276ff6_b43e9755eaf7":0.9983288050769804,"15d4ac2e73f7_3229ea47d96d":0.0005645267344228102,"15d4ac2e73f7_d84df75bbf5f":0.9942812284113284,"15e05b968693_787fc9df6e2f":0.9951411486342134,"1901f9b561d4_2ed57bcc51d8":0.9985089897334577,"1b0817b8df7e_2ba4bdbd8fea":0.0002480212009577893,"1c4376ec2835_24ddf49ad519":0.9841972349008037,"1c4376ec2835_af09fe488ef1":0.013906870575706,"1c4376ec2835_d2564590ec5f":0.9847105398968459,"1c4376ec2835_e4efdd3f271b":0.0003504272294065954,"1d71ae73298c_d84df75bbf5f":0.0003916734609645605,"1e5a9ae7d77e_fc0920e0882b":0.9961866618362232,"1f48cc065ef7_eae579e5f473":0.9981829971133477,"22884f37fb56_c85c85cd1759":0.00013181151697447486,"24ddf49ad519_e4efdd3f271b":0.0003893757071571015,"2633aaf73fa5_8d6a40a8796c":0.0007131810021376653,"272473d519f1_3229ea47d96d":0.00035483173825544853,"272473d519f1_d84df75bbf5f":0.00035557197565784434,"275a2e391bfb_95de0e29a8c9":0.00039043802275181514,"27a7446358a7_f6e3a2eb6838":0.0005316548300369615,"28d4a612b00f_3d87d3d5f636":0.00042422268464623443,"29b94caf7c86_5cd719d33027":0.0014101274978884,"2b75a7e57c3d_5b85d9213321":0.9984783987259798,"2b75a7e57c3d_aeb6a71ed2ab":0.00026036535790447665,"2cb47272cf9c_647c9065bbeb":0.9984146629158376,"3091224407dc_8b6b603e240c":0.00011134071751430357,"31ae7668a70d_ae5b5f4ccfff":0.0003178449677057576,"324b771367e0_78471db61880":0.0003449304592612928,"324b771367e0_eae579e5f473":0.00023880363392859546,"34830c3d7c74_d420d1e0b489":0.0006742473754803032,"38f93047fa27_fcebb06c2a34":0.9984009064803333,"4016726791c4_da994e881861":0.001415374555625119,"41fe0bd6d982_fee6c2b435ed":0.7011454610980521,"4462504cc1e5_b01b7f1151e9":0.00019716528604214457,"4b716da37ec2_7445dc20376f":0.14835467242229772,"4cbc2460f2c7_b3795571e73d":0.9953866456144891,"4fc44c6075f8_9ec8389cc58c":0.9984058138216999,"54d61596d300_81f3471e70d1":0.0019528526361416141,"5ab493325768_63fae857b3b6":0.0014851890302548547,"615bddaa6122_71cd1ff7aa79":0.9970344090746976,"61edaef48537_654e9478b215":0.0007712068876926405,"61edaef48537_66ced1826432":0.9890239123239286,"654e9478b215_bb82a255c1a8":0.0011566608009016166,"668e4ff994f2_dd71b6776cd3":0.0029121838214830427,"66ced1826432_c96d506c9025":0.0004951072431028037,"66f21c337e4d_ec8877680f5b":0.0002713038265709989,"685a7b710161_71cd1ff7aa79":0.0004379777237637235,"68d0de9e4b94_8bee00777572":0.00028181312600553864,"6b6833baf993_85cdc8e62bc8":0.00041775597847134476,"6b6833baf993_ad524e4cefd7":0.0017027767652924946,"6b6833baf993_b9e07208e34f":0.01492500031814824,"6cbeab253384_ec90a71db921":0.0017444445155435065,"6eafab761f93_be7ed49013a3":0.00033317552505855,"702bcf110484_ea50be3b0404":0.0004901625880314563,"705ee1ba948c_790a109f0a84":0.9985931802283478,"71cd1ff7aa79_e853be22014d":0.0005635021020986805,"722bbf2c3cec_8572a5448c4c":0.00015819800574954864,"787fc9df6e2f_941127cf2788":0.008816249439579189,"7dd67591e9d9_a6b5762426cd":0.0016701896421010597,"7dd67591e9d9_ec90a71db921":0.00317884769471765,"7eee88e0f094_b5ddee2f3067":0.7207400325066723,"8086ad9f717e_b3795571e73d":0.0005376211405868524,"8158fe6ee2bf_ba1fd9f1beea":0.00028935198214841894,"81da04a4f4ce_e78763e1c5a4":0.000257015995006097,"84fc58b45fa9_afb63d20dafc":0.004437151021819941,"84fe6219552b_e8343e4a08b2":0.00126404865985045,"85cdc8e62bc8_ad524e4cefd7":0.001346834910115009,"85cdc8e62bc8_b9e07208e34f":0.008861415725336306,"85cdc8e62bc8_d2a2dcd4fe04":0.0002690709524676765,"89e2901f53a4_af09fe488ef1":0.996420942422416,"8d7cf7dbe653_b7a8156640ac":0.0002611006853171841,"93e2adb4dc8a_22c80596622a":0.9927643595478962,"941127cf2788_950c3dc6c2ff":0.9976112139408656,"941127cf2788_af09fe488ef1":0.0007655862615461343,"94effc305bb2_a7423b338c13":0.9982499965022829,"97885c30e13b_d01f3e239423":0.0005384047188107766,"979db675dc53_a0c41c8b18ca":0.0889758598662219,"98144a291186_c90b246edeaa":0.0002266635537207886,"984d613009c2_647c9065bbeb":0.0009340713063919694,"984d613009c2_6b6982605998":0.0003610929676352388,"9a3d06d8544e_fee6c2b435ed":0.9840266512933745,"a0c41c8b18ca_ec90a71db921":0.0024329511673844106,"a34a1f1d8475_b173b5fd3a05":0.997570559862718,"a34a1f1d8475_d2564590ec5f":0.00047544149485226307,"a472ace6672b_01d7b8d31ae0":0.0003374340361253262,"a472ace6672b_3c64c33d040e":0.0003392603883287679,"a472ace6672b_60a3c410247d":0.0002057169009479425,"a472ace6672b_f2da92ef3bb7":0.0024981295265206226,"a7411a79c315_8d4f7b102f48":0.04207704048330388,"a9fab67d78d4_ecfe5bb98408":0.00017856918869766026,"ad524e4cefd7_d2a2dcd4fe04":0.0003733936989377275,"ad9b0de8c030_3bdddcf4b05f":0.9706444852269801,"ad9b0de8c030_b05e7bf9b66c":0.8533694288415453,"afb63d20dafc_040c9e16c0f1":0.00030360201964214407,"b173b5fd3a05_c88ca6b3ac3e":0.0003530487921808839,"b173b5fd3a05_d2564590ec5f":0.00031237450479066253,"b7cba8280202_bf6961332e66":0.00033676108146810243,"b9e07208e34f_d2a2dcd4fe04":0.000585910632376904,"c35765e63518_d420d1e0b489":0.000201256930317927,"c55961beed26_dcf77a9bcbca":0.0006969864678514369,"c88ca6b3ac3e_d2564590ec5f":0.00022773714974074334,"ca63076678b9_e66e6faff6c4":0.000699649807691637,"ca63076678b9_ec90a71db921":0.25709383373219796,"cc7d6865340e_f7fcb3b2497d":0.0006825188199208769,"d01f3e239423_e74fd2ce0e55":0.9984838195540101,"d1a2cd3c420b_fcebb06c2a34":0.00036530397958038196,"e383ade6a43d_e66e6faff6c4":0.9968536946936724,"e8718b6a3653_eb0c309a0960":0.0010461863850215736,"ecfe5bb98408_ef598da0a6e4":0.006515115638609257,"faea8056ba3b_4672b69e4c2a":0.9979290549850639}""")

def llm_verify_fp32_against_committed():
    """Protocol 5.11 gate: refuses (raises) rather than continuing on a silent divergence --
    no V2/V3 built from an unverified export is trusted."""
    run_worker("llm-verify-fp32", kind="llm", name="llm", variant="onnxfp32-verify", model=str(S["llm_fp32"]),
               tokenizer=str(S["llm_merged"] / "tokenizer.json"), pad_id=tokenizer.pad_token_id, pin=False,
               score=score_spec("llm", {"val": LLM_SETS["val"]}, "onnxfp32-verify"))
    fresh = json.loads((OUT / "preds-llm-onnxfp32-verify-val.json").read_text())
    common = sorted(set(fresh) & set(COMMITTED_LLM_ONNXFP32_VAL))
    if not common:
        raise SystemExit("REFUSING: no overlap between fresh and committed val pair_ids")
    if not set(fresh) <= set(COMMITTED_LLM_ONNXFP32_VAL):
        raise SystemExit("REFUSING: fresh val predictions cover pair_ids absent from the committed file")
    diffs = {k: abs(fresh[k] - COMMITTED_LLM_ONNXFP32_VAL[k]) for k in common}
    max_diff = max(diffs.values())
    jupdate("model-facts.json", "llm_fp32_verify_vs_committed",
            {"n": len(common), "max_abs_diff": max_diff, "smoke_subset": SMOKE})
    print(f"LLM fp32 verify: max|diff| vs committed = {max_diff:.2e} over {len(common)} pairs (SMOKE={SMOKE})")
    if max_diff > 1e-5:
        raise SystemExit(
            f"REFUSING: fresh LLM ONNX fp32 export diverges from the committed reference by "
            f"{max_diff:.2e} > 1e-5 -- protocol 5.11 gate failed, int8 variants built from this "
            "export cannot be trusted."
        )

def llm_int8_v2():
    dst = TMP / "llm-int8-v2" / "model.onnx"
    quantize(S["llm_fp32"], dst, external=True, per_channel=True, reduce_range=True)
    S["llm_int8_v2"] = dst
    jupdate("model-facts.json", "llm_onnx_int8_v2", {"bytes": dir_bytes(dst.parent), "sha256_graph": sha256_file(dst),
                                                    "quantization": "onnxruntime quantize_dynamic, QInt8 weights, per_channel=True, reduce_range=True"})

def llm_int8_v3():
    dst = TMP / "llm-int8-v3" / "model.onnx"
    quantize(S["llm_fp32"], dst, external=True, per_channel=True, reduce_range=True, op_types_to_quantize=["MatMul"])
    S["llm_int8_v3"] = dst
    jupdate("model-facts.json", "llm_onnx_int8_v3", {"bytes": dir_bytes(dst.parent), "sha256_graph": sha256_file(dst),
                                                    "quantization": "onnxruntime quantize_dynamic, QInt8 weights, per_channel=True, reduce_range=True, op_types_to_quantize=[MatMul] only"})

def llm_score_variant(tag, key):
    run_worker(f"llm-score-{tag}", kind="llm", name="llm", variant=tag, model=str(S[key]),
               tokenizer=str(S["llm_merged"] / "tokenizer.json"), pad_id=tokenizer.pad_token_id, pin=False,
               score=score_spec("llm", LLM_SETS, tag))

def llm_latency_variant(tag, key):
    run_worker(f"llm-bench-{tag}", kind="llm", name="llm", variant=tag, model=str(S[key]),
               tokenizer=str(S["llm_merged"] / "tokenizer.json"), pad_id=tokenizer.pad_token_id, pin=True,
               bench=bench_spec("llm", "llm", tag))
    lat = json.loads((OUT / f"latency-llm-{tag}.json").read_text())
    jupdate("throughput.json", f"llm-{tag}", lat["throughput"])
    jupdate("model-facts.json", f"llm_{tag}_peak_rss_mb", lat["peak_rss_mb"])

run_stage("LLM-merge", llm_merge)
run_stage("LLM-export", llm_export)
run_stage("LLM-verify-fp32", llm_verify_fp32_against_committed)
if not STATUS[-1]["ok"]:
    # run_stage() catches SystemExit too (so every stage always gets a status row) --
    # that means llm_verify_fp32_against_committed's own `raise SystemExit` on divergence
    # would otherwise just be recorded as ok=false and the notebook would carry on
    # quantizing V2/V3 from an unverified export. Re-raise HERE, outside run_stage, so a
    # failed gate actually stops "Run All" (protocol 5.11; runbook section E.4).
    raise SystemExit(
        "LLM-verify-fp32 FAILED -- refusing to quantize/score V2/V3 from an unverified "
        f"export: {STATUS[-1]['error']}"
    )
run_stage("LLM-int8-v2", llm_int8_v2)
run_stage("LLM-score-v2", lambda: llm_score_variant("int8-v2", "llm_int8_v2"))
run_stage("LLM-latency-v2", lambda: llm_latency_variant("int8-v2", "llm_int8_v2"))
run_stage("LLM-int8-v3", llm_int8_v3)
run_stage("LLM-score-v3", lambda: llm_score_variant("int8-v3", "llm_int8_v3"))
run_stage("LLM-latency-v3", lambda: llm_latency_variant("int8-v3", "llm_int8_v3"))


In [ ]:
jdump(STATUS, "stage-status.json")
print(json.dumps(STATUS, indent=1))
print("SMOKE" if SMOKE else "FULL", "run finished. Outputs under", OUT)
from IPython.display import FileLink, display
for p in sorted(OUT.rglob("*.json")):
    display(FileLink(str(p.relative_to(WORK))))